<a href="https://colab.research.google.com/github/MarcosRigal/AP/blob/main/adapted_performance_metrics_for_oc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparación de MAE y TC vs. MAE_int y TC_int

En este notebook se ilustra de forma práctica lo presentado en el paper:

- Se definen las métricas ordinales **MAE** y **TC** a partir de la matriz de confusión.
- Se adaptan estas métricas a escala intervalar (**MAE_int** y **TC_int**) usando la distancia entre intervalos y el número de instancias sobre la longitud del intervalo.
- Se estudia el efecto de modificar la longitud del último intervalo mediante un barrido de valores para *x*.

Se usan dos matrices de confusión simuladas con tres clases y un conjunto de datos balanceado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 5)

def mae_ordinal(conf_matrix):
    """
    Calcula MAE ordinal = (1/N) * sum(conf[i,j] * |i - j|).
    conf_matrix: r x r, filas = predicción, columnas = valor verdadero.
    """
    N = np.sum(conf_matrix)
    r = conf_matrix.shape[0]
    indices = np.arange(r)
    diff = np.abs(indices.reshape(-1,1) - indices.reshape(1,-1))
    return np.sum(conf_matrix * diff) / N

def tc_ordinal(conf_matrix):
    """
    Calcula TC ordinal, introducido en George et al. (2016).
    TC = sum_{i,j} conf[i,j] * ((N - n_j)/n_i) * |i - j|
      - n_j = número de instancias en la clase verdadera j
      - n_i = número de instancias asignadas a la clase i
    Penaliza más los errores en clases pequeñas.
    """
    N = np.sum(conf_matrix)
    r = conf_matrix.shape[0]
    row_sums = np.sum(conf_matrix, axis=1)
    col_sums = np.sum(conf_matrix, axis=0)
    total_cost = 0.0
    for i in range(r):
        for j in range(r):
            if i != j:
                weight = (N - col_sums[j]) / row_sums[i]
                total_cost += conf_matrix[i,j] * weight * abs(i - j)
    return total_cost

def hausdorff_distance(interval_a, interval_b):
    """
    Distancia de Hausdorff para intervalos 1D:
    d([a_i, b_i], [a_j, b_j]) = max(|a_j - a_i|, |b_j - b_i|).
    """
    a_i, b_i = interval_a
    a_j, b_j = interval_b
    return max(abs(a_j - a_i), abs(b_j - b_i))

def mae_interval(conf_matrix, intervals):
    """
    MAE_int = (1/N) * sum_{i,j} conf[i,j] * d(interval_i, interval_j)
    Donde d es la distancia (aquí Hausdorff) entre intervalos.
    """
    N = np.sum(conf_matrix)
    r = conf_matrix.shape[0]
    total = 0.0
    for i in range(r):
        for j in range(r):
            dist = hausdorff_distance(intervals[i], intervals[j])
            total += conf_matrix[i,j] * dist
    return total / N

def tc_interval(conf_matrix, intervals, class_counts):
    """
    Versión intervalar de TC:
      - L_j = longitud(I_j)
      - densidad rho_j = n_j / L_j
      - costo(i->j) = ((sum_{k != j} rho_k) / rho_i) * d(I_i, I_j)
    """
    r = conf_matrix.shape[0]
    lengths = np.array([interval[1] - interval[0] for interval in intervals])
    densities = class_counts / lengths
    sum_densities = np.sum(densities)
    total = 0.0
    for i in range(r):
        for j in range(r):
            if i != j:
                dist = hausdorff_distance(intervals[i], intervals[j])
                weight = (sum_densities - densities[j]) / densities[i]
                total += conf_matrix[i,j] * weight * dist
    return total



### Sección 1: Métricas Ordinales vs. Intervalares
Ejemplo con matrices de confusión para 3 clases.

In [ ]:
conf_A = np.array([[3,2,1],
                   [2,2,2],
                   [0,1,2]])
conf_B = np.array([[3,2,2],
                   [2,2,1],
                   [0,1,2]])

true_counts = np.array([5,5,5])
N = np.sum(true_counts)
r = 3

print("### Métricas Ordinales ###")
print("MAE(conf_A):", mae_ordinal(conf_A))
print("MAE(conf_B):", mae_ordinal(conf_B))
print("TC(conf_A): ", tc_ordinal(conf_A))
print("TC(conf_B): ", tc_ordinal(conf_B))

intervals = [(0,1), (1,2), (2,3)]

mae_A_int = mae_interval(conf_A, intervals)
mae_B_int = mae_interval(conf_B, intervals)
tc_A_int = tc_interval(conf_A, intervals, true_counts)
tc_B_int = tc_interval(conf_B, intervals, true_counts)

print("\n### Métricas Intervalares ###")
print("MAE_int(conf_A):", mae_A_int)
print("MAE_int(conf_B):", mae_B_int)
print("TC_int(conf_A): ", tc_A_int)
print("TC_int(conf_B): ", tc_B_int)


### Sección 2: Barrido de `x` (longitud del último intervalo)
En lugar de tomar el intervalo `I3=[2,3]`, probamos varios valores: `I3=[2, 2+x]`.

In [ ]:
def define_intervals(x):
    return [(0,1),(1,2),(2,2+x)]

x_vals = np.linspace(0.5, 2.0, 20)
mae_A_int_vals, mae_B_int_vals = [], []
tc_A_int_vals,  tc_B_int_vals  = [], []

for x in x_vals:
    ints = define_intervals(x)
    mae_A_int_vals.append(mae_interval(conf_A, ints))
    mae_B_int_vals.append(mae_interval(conf_B, ints))
    tc_A_int_vals.append(tc_interval(conf_A, ints, true_counts))
    tc_B_int_vals.append(tc_interval(conf_B, ints, true_counts))

print("Barrido x de 0.5 a 2.0 en 20 pasos (primeros 5 resultados):\n")
for i in range(5):
    x = x_vals[i]
    print(f"x={x:.2f} -> MAE_int(A)={mae_A_int_vals[i]:.3f}, TC_int(A)={tc_A_int_vals[i]:.3f}")

### Sección 3: Gráficas de la evolución de las métricas intervalares vs. `x`

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(x_vals, mae_A_int_vals, marker='o', label="MAE_int(A)")
plt.plot(x_vals, mae_B_int_vals, marker='s', label="MAE_int(B)")
plt.title("MAE_int vs. x")
plt.xlabel("x (longitud de I3)")
plt.ylabel("MAE_int")
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(x_vals, tc_A_int_vals, marker='o', label="TC_int(A)")
plt.plot(x_vals, tc_B_int_vals, marker='s', label="TC_int(B)")
plt.title("TC_int vs. x")
plt.xlabel("x (longitud de I3)")
plt.ylabel("TC_int")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Discusión breve

- Las métricas ordinales (MAE, TC) son constantes respecto a `x` y no tienen en cuenta la longitud de los intervalos.
- Las métricas intervalares (MAE_int, TC_int) sí dependen de la longitud, de modo que, conforme `x` crece, cambia la penalización por errores en la clase `I3`.
- Este cuaderno ilustra la diferencia conceptual y práctica entre métricas ordinales y sus versiones de escala intervalar.


## Sección 7: Ejemplo con datos simulados y un clasificador real
En esta sección construimos un ejemplo **paso a paso**:
1. Generamos datos sintéticos donde haya una variable "Edad" (años) y dos "features" adicionales.
2. Discretizamos la edad en 3 intervalos: `[0, 30), [30, 60), [60, ∞)`.
3. Entrenamos un modelo de clasificación (RandomForest) para predecir en cuál de esos intervalos cae la edad.
4. Obtenemos la matriz de confusión y calculamos nuestras métricas.
5. Como "no está acotado" el intervalo `[60,∞)`, elegimos una cota.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

np.random.seed(42)
N_synth = 300

X1 = np.random.normal(loc=0, scale=1, size=N_synth)
X2 = np.random.normal(loc=5, scale=2, size=N_synth)

edad = np.random.uniform(low=10, high=100, size=N_synth)

y = np.zeros(N_synth, dtype=int)
y[(edad >= 30) & (edad < 60)] = 1
y[edad >= 60] = 2

X = np.column_stack((X1, X2))

X_train, X_test, y_train, y_test, edad_train, edad_test = train_test_split(X, y, edad,
                                                    test_size=0.3,
                                                    random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("Matriz de Confusión (RandomForest):\n", cm)

MAE_model = mae_ordinal(cm)
TC_model  = tc_ordinal(cm)
print(f"\nMAE ordinal = {MAE_model:.3f}")
print(f"TC ordinal  = {TC_model:.3f}")

counts_test = np.array([np.sum(y_test==0), np.sum(y_test==1), np.sum(y_test==2)])
print("\nClases verdaderas en el test:", counts_test)

intervals_real = [(0,30), (30,60), (60,90)]

MAE_int_model = mae_interval(cm, intervals_real)
TC_int_model  = tc_interval(cm, intervals_real, counts_test)

print(f"\nMAE_int = {MAE_int_model:.3f}")
print(f"TC_int  = {TC_int_model:.3f}")